# 实训大作业 · Notebook：把课程提供的 SwanRmsNorm 算子接到 Qwen3-8B

本 Notebook 是 [`guidance.pdf`](pdf/guidance.pdf) 学生操作指导书 的实操配套。

**前提**：课程提供的 SwanRmsNorm AscendC kernel 已经在指导书 §5 编译并部署到 NPU 上。本 Notebook 只做**集成层 + 端到端实验**，分三步：

1. **写集成层** — 用 `torch.autograd.Function` 包装 SwanRmsNorm 的 forward / backward，并 monkey-patch 到 Qwen3 模型里
2. **测推理加速** — 对比 baseline / patched 的 tokens/sec，画图
3. **跑 LoRA SFT** — 用 alpaca_zh 数据集做一轮微调，对比训练 step 时间，并把 loss + step time 用 SwanLab 记录起来

**前置条件**：
- 已经按 [guidance.pdf](pdf/guidance.pdf) §5 把课程提供的 `swan_rms_norm` AscendC kernel 编译并安装到 `~/Ascend/custom_opp`
- 启动 Jupyter 之前已在 shell 里 `source ~/Ascend/custom_opp/vendors/xllm/bin/set_env.bash`（否则 §1 中 `import rms_norm_lib` 会失败）
- Python 依赖与 Qwen3-8B 基座模型由下面的 §0 自动处理，无需手动准备

**Notebook 内有 2 个 `# TODO` 需要你补全**——参考 [guidance.pdf](pdf/guidance.pdf) §6 完成。直接运行会在 TODO 处 `NotImplementedError`。

> **注意**：你**不需要**写 AscendC kernel——那是课程已经独立实现好的。你的任务是把它接到 PyTorch 与 Qwen3 模型上。

---

## §0 环境准备

本 Notebook 在 **CANNLab 的 Ascend NPU 环境**下运行。镜像默认已预装 `torch` 与 `torch_npu`，下面两步分别处理：

1. **Python 上层依赖**：用 `pip` 装 `transformers` / `peft` / `datasets` / `swanlab` / `accelerate` / `modelscope`
2. **Qwen3-8B 基座模型**：从 ModelScope 下载到 `./Qwen3-8B`（约 16GB，仅首次需要，10-20 分钟）

> **首次运行**：直接顺序执行下方两个 cell 即可。
>
> **重跑场景**：如果环境里已经残留过旧版 `transformers` 且本 Notebook 之前 import 过它，安装后请点 `Kernel → Restart`，否则新版本不会生效。

> **⚠️ 重要前提**：自定义算子 `rms_norm_lib` **无法通过 pip 安装**，必须按 [guidance.pdf](pdf/guidance.pdf) §5 在 shell 里手动编译并安装到 `~/Ascend/custom_opp`，并在启动 Jupyter 之前 `source ~/Ascend/custom_opp/vendors/xllm/bin/set_env.bash`。否则 §1 环境检查中的 `import rms_norm_lib` 会失败。

In [ ]:
# 1. 安装 Python 上层依赖
%pip install -q -i https://pypi.tuna.tsinghua.edu.cn/simple modelscope==1.35.4 transformers==5.5.4 peft==0.18.0 datasets==4.8.4 swanlab==0.7.15 accelerate==1.13.0

In [ ]:
# 2. 下载 Qwen3-8B 基座模型（首次约 16GB / 10-20 分钟，已下载则跳过）
import os

MODEL_DIR = './Qwen3-8B'
if os.path.exists(MODEL_DIR) and os.listdir(MODEL_DIR):
    print(f'{MODEL_DIR} 已存在，跳过下载')
else:
    !modelscope download --model Qwen/Qwen3-8B --local_dir ./Qwen3-8B

---

## §1 环境检查

确认 NPU 可用、torch_npu 装好、自定义算子能 import。任何一行报错都得回到指导书 §4-§5 排查。

In [ ]:
import time, gc, types
import torch
import torch_npu

print(f'torch     : {torch.__version__}')
print(f'torch_npu : {torch_npu.__version__}')
assert torch.npu.is_available(), 'NPU 不可用！'
torch_npu.npu.set_device(0)
print(f'device    : {torch_npu.npu.get_device_name(0)}')

import rms_norm_lib
print(f'rms_norm_lib loaded: {rms_norm_lib}')
print(f'available functions: {[x for x in dir(rms_norm_lib) if not x.startswith("_")]}')

---

## §2 算子精度验证

在调用模型前先确认我们的算子计算正确。和 PyTorch native 实现对比，bf16 + 随机 γ 的实测最大绝对误差通常落在 **0.1-0.4** 范围内，超过 0.5 才是真有问题。

我们用 Qwen3-8B 的 `hidden_size = 4096` 来测，更接近真实场景的形状。

In [ ]:
torch.manual_seed(42)

# Qwen3-8B 的 hidden_size = 4096
M, N = 16, 4096
x = torch.randn(M, N, dtype=torch.bfloat16, device='npu')
gamma = torch.randn(N, dtype=torch.bfloat16, device='npu')

# 我们的算子
y_swan = rms_norm_lib.rms_norm(x, gamma, 1e-6)

# PyTorch native（同 Qwen3RMSNorm.forward）
xf = x.to(torch.float32)
y_torch = (gamma * (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + 1e-6)).to(x.dtype))

diff = (y_swan.float() - y_torch.float()).abs().max().item()
print(f'shape       : {tuple(y_swan.shape)}, dtype: {y_swan.dtype}')
print(f'max |diff|  : {diff:.4f}   (期望 < 0.5；典型范围 0.1-0.4)')
assert diff < 0.5, '精度不对，回去检查 kernel 实现'

---

## §3 加载 Qwen3-8B

用之前已经下载的 Qwen3-8B 模型。后面所有测试都会重复加载这个模型——8B 在 bf16 下 ~16GB，HBM 60GB 够用，但每次 load 大约 10-30 秒。

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.qwen3 import modeling_qwen3

MODEL_PATH = './Qwen3-8B'   # ~ 自动解析当前用户家目录

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 数一下模型里有多少个 Qwen3RMSNorm 模块
_tmp = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.bfloat16)
n_rmsnorm = sum(1 for m in _tmp.modules() if isinstance(m, modeling_qwen3.Qwen3RMSNorm))
n_layers  = len(_tmp.model.layers)
print(f'Qwen3-8B:')
print(f'  layers          : {n_layers}')
print(f'  Qwen3RMSNorm    : {n_rmsnorm}  (= {n_layers} 层 × 4/层 + 1 final)')
print(f'  hidden_size     : {_tmp.config.hidden_size}')
del _tmp; gc.collect(); torch.npu.empty_cache()

---

## §4 定义 SwanRmsNormFunction（autograd.Function）

**TODO #1**：参考指导书 §6.2.2，补全下面的 `forward` 和 `backward`。

要点：
- **forward** 调 `rms_norm_lib.rms_norm(x, gamma, eps)`，并通过 `ctx` 保存 backward 需要的张量
- **backward** 用 `torch_npu.npu_rms_norm_backward(grad_y, x, gamma, rstd)`，注意 `rstd` 必须是 fp32 且要从 `x` 重算
- 返回值数量必须与 forward 形参一致——没有梯度的参数返回 `None`

In [ ]:
class SwanRmsNormFunction(torch.autograd.Function):
    """
    Forward: 用我们的 SwanRmsNorm AscendC kernel
    Backward: 用 torch_npu 的 fused npu_rms_norm_backward
    """

    @staticmethod
    def forward(ctx, x, gamma, eps):
        # ───────── TODO #1 (a) ─────────
        # 1) x.contiguous() 一下（kernel 假设 stride 对齐，stride 不对会 silently 算错）；
        # 2) 调 rms_norm_lib.rms_norm(x_contig, gamma, eps) 算 forward；
        # 3) 用 ctx.save_for_backward(x_contig, gamma) 保存反向需要的张量；
        # 4) eps 是 float（不是 tensor），存到 ctx.eps 上。
        # 详见指导书 §6.2.2。
        raise NotImplementedError('请补全 forward')

    @staticmethod
    def backward(ctx, grad_y):
        x, gamma = ctx.saved_tensors
        eps = ctx.eps
        # ───────── TODO #1 (b) ─────────
        # 用 torch_npu.npu_rms_norm_backward(grad_y, x, gamma, rstd) 算梯度。
        # rstd 不在 ctx 里，需要从 x 重算（fp32 精度，否则报 dtype 错）。
        # 返回 (grad_x, grad_gamma, None) ——eps 没有梯度。
        # 详见指导书 §6.2.2。
        raise NotImplementedError('请补全 backward')


# 简单 forward 验证
y_check = SwanRmsNormFunction.apply(x, gamma, 1e-6)
diff = (y_check.float() - y_torch.float()).abs().max().item()
print(f'autograd.Function forward max |diff| = {diff:.4f}')
assert diff < 0.5

---

## §5 实现 patch_qwen3_rmsnorm（带计数）

**TODO #2**：参考指导书 §6.2.3，补全下面的 patch 函数。

要点：
- 把这个 module 的 `forward` 替换为调 `SwanRmsNormFunction.apply` 的版本
- 每个 module 有自己的 `weight`，需要用**工厂函数**捕获——避免 Python 闭包后期绑定让所有 module 都用最后一个 weight
- 替换时用 `types.MethodType(...)` 让函数能正确接收 `self`
- 别忘了 `n += 1` 计数

In [ ]:
def patch_qwen3_rmsnorm(model):
    n = 0
    for module in model.modules():
        if isinstance(module, modeling_qwen3.Qwen3RMSNorm):
            eps = module.variance_epsilon
            weight = module.weight

            # ───────── TODO #2 ─────────
            # 1) 写一个工厂函数 make_fw(w, e) 返回新 forward
            # 2) 用 types.MethodType 把它绑定到 module
            # 3) n += 1
            # 详见指导书 §6.2.3。
            raise NotImplementedError('请补全 patch 逻辑')
    return n


# 验证 patch
_test_model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.bfloat16).to('npu').eval()
n_patched = patch_qwen3_rmsnorm(_test_model)
print(f'patched {n_patched} Qwen3RMSNorm modules  (期望: {n_rmsnorm})')
assert n_patched == n_rmsnorm, 'patch 数量不对'
del _test_model; gc.collect(); torch.npu.empty_cache()

---

## §6 推理加速测试

测 Qwen3-8B 的端到端推理 tokens/sec。Baseline / Patched 各加载一次模型，避免相互污染。

In [ ]:
PROMPT = '请用一段话介绍量子计算的基本原理。'
MAX_NEW = 50

inputs = tokenizer(PROMPT, return_tensors='pt').to('npu')

def measure_tps(model, max_new=MAX_NEW, warmup=2):
    """测一次 generate 的 tokens/sec"""
    for _ in range(warmup):
        _ = model.generate(**inputs, max_new_tokens=max_new, do_sample=False,
                           pad_token_id=tokenizer.eos_token_id)
    torch_npu.npu.synchronize()
    t0 = time.perf_counter()
    out = model.generate(**inputs, max_new_tokens=max_new, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    torch_npu.npu.synchronize()
    dt = time.perf_counter() - t0
    n_new = out.shape[1] - inputs['input_ids'].shape[1]
    return n_new / dt, tokenizer.decode(out[0], skip_special_tokens=True)


# Baseline
print('===== Baseline (PyTorch native RMSNorm) =====')
m_base = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.bfloat16).to('npu').eval()
tps_baseline, text_baseline = measure_tps(m_base)
print(f'  tokens/sec: {tps_baseline:.2f}')
del m_base; gc.collect(); torch.npu.empty_cache()

# Patched
print('\n===== Patched (SwanRmsNorm AscendC kernel) =====')
m_pt = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.bfloat16).to('npu').eval()
n = patch_qwen3_rmsnorm(m_pt)
print(f'  patched {n} modules')
tps_patched, text_patched = measure_tps(m_pt)
print(f'  tokens/sec: {tps_patched:.2f}')
del m_pt; gc.collect(); torch.npu.empty_cache()

# 总结
speedup = tps_patched / tps_baseline
print('\n' + '═' * 50)
print(f'  Baseline       : {tps_baseline:.2f} tokens/sec')
print(f'  Patched        : {tps_patched:.2f} tokens/sec')
print(f'  Speedup        : {speedup:.3f} ×')
print('═' * 50)
print(f'\nGreedy decode 一致性：{text_baseline == text_patched}')

## §6.1 推理加速可视化

In [ ]:
import os
import matplotlib
matplotlib.use('Agg')   # headless backend
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

# 左图：tokens/sec 对比
ax1.bar(['Baseline\n(PyTorch native)', 'Patched\n(SwanRmsNorm)'],
        [tps_baseline, tps_patched],
        color=['#9E9E9E', '#4CAF50'])
ax1.set_ylabel('tokens / sec')
ax1.set_title('Qwen3-8B Inference Throughput')
ax1.text(1, tps_patched, f'{speedup:.2f}×', ha='center', va='bottom',
         fontsize=14, fontweight='bold', color='#2E7D32')

# 右图：speedup 单柱
ax2.bar(['Speedup'], [speedup], color='#4CAF50', width=0.5)
ax2.axhline(1.0, color='gray', linestyle='--', linewidth=1)
ax2.set_ylabel('Speedup (×)')
ax2.set_title('End-to-end Speedup')
ax2.text(0, speedup, f'{speedup:.3f}×', ha='center', va='bottom',
         fontsize=14, fontweight='bold', color='#2E7D32')
ax2.set_ylim(0, max(2, speedup * 1.2))

plt.tight_layout()
plt.savefig('notebook_inference.png', dpi=120)
plt.show()
print('保存到 notebook_inference.png')

---

## §7 LoRA 微调：使用 alpaca_zh 数据集

下面用 alpaca_zh 数据集对 Qwen3-8B 做一轮 LoRA 微调，对比 baseline / patched 的训练速度。

### §7.1 下载并加载 alpaca_zh

In [ ]:
import os
if not os.path.exists('alpaca_zh'):
    print('下载 alpaca_zh 数据集...')
    !modelscope download --dataset llamafactory/alpaca_zh --local_dir ./alpaca_zh
else:
    print('alpaca_zh 已存在，跳过下载')

import datasets
alpaca = datasets.load_dataset('json', data_files='alpaca_zh/alpaca_data_zh_51k.json')['train']
print(f'数据集样本数: {len(alpaca)}')
print('\n第一条样本:')
print(f'  instruction: {alpaca[0]["instruction"]}')
print(f'  input      : {alpaca[0]["input"]}')
print(f'  output     : {alpaca[0]["output"][:80]}...')

### §7.2 数据格式化

把 instruction/input/output 拼成 Qwen3 chat 格式：

In [ ]:
def format_data(example):
    user_text = example['instruction'] + (example['input'] or '')
    assistant_text = example['output']
    messages = [
        {'role': 'user', 'content': user_text},
        {'role': 'assistant', 'content': assistant_text},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False, enable_thinking=False)
    return {'text': text}

# 取 200 条做演示（实测 alpaca_zh 全集 51k 单 epoch 太长）
DEMO_SIZE = 200
demo_data = alpaca.select(range(DEMO_SIZE)).map(format_data, remove_columns=alpaca.column_names)
print(f'演示用数据集大小: {len(demo_data)} 条')
print(f'\n格式化后第一条:\n{demo_data[0]["text"][:300]}...')

### §7.3 构造 LoRA 模型 + 测一步训练时间（同时用 SwanLab 记录）

跑 30 步 LoRA SFT，每步把 loss 和 step_time_ms 推到 SwanLab。两个变体（baseline / patched）独立 run，可以在 SwanLab 上叠加对比。

> **使用 SwanLab 前**：在 shell 里跑 `swanlab login`，输入你的 API key（[swanlab.cn/settings](https://swanlab.cn/settings)）。如果没登录，cell 里有 try/except 兜底，跟踪会跳过但测速本身不受影响。

> ⏳ **训练耗时较长**（baseline + patched 共两轮 LoRA SFT，约 10-20 分钟）。**点击运行后等 cell 跑完再点下一个**——SwanLab 训练 cell 不会自动跳转，连续点会重复触发训练。

In [ ]:
from peft import LoraConfig, get_peft_model

BATCH_SIZE = 4
SEQ_LEN = 512
LORA_R = 16
N_TRAIN_STEPS = 30   # 跑 30 步看时间趋势（前 5 步 warmup）
WARMUP = 5

def build_lora_model():
    base = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.bfloat16)
    for p in base.parameters():
        p.requires_grad = False
    cfg = LoraConfig(
        r=LORA_R, lora_alpha=LORA_R*2,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_dropout=0.0, bias='none', task_type='CAUSAL_LM',
    )
    return get_peft_model(base, cfg).to('npu')

def make_batches(n):
    """取 n 个 batch（每个 batch_size 条），固定 padding 到 seq_len。
    label 里的 pad 位置设为 -100 → CrossEntropy 会忽略，loss 才能正常下降。"""
    batches = []
    for i in range(n):
        texts = [demo_data[(i * BATCH_SIZE + j) % len(demo_data)]['text']
                 for j in range(BATCH_SIZE)]
        enc = tokenizer(texts, return_tensors='pt', truncation=True,
                        padding='max_length', max_length=SEQ_LEN).to('npu')
        labels = enc['input_ids'].clone()
        labels[enc['attention_mask'] == 0] = -100   # 不要 train pad token
        enc['labels'] = labels
        batches.append(enc)
    return batches

batches = make_batches(N_TRAIN_STEPS)
print(f'构造 {N_TRAIN_STEPS} 个 batch, batch_size={BATCH_SIZE}, seq_len={SEQ_LEN}')
print(f'M = batch × seq_len = {BATCH_SIZE * SEQ_LEN}')

In [ ]:
# ====== SwanLab 登录 + 配置 ======
import getpass
import swanlab
from pathlib import Path

# 项目名统一为 CANN_SwanLab，便于和前两章实验放在同一项目下
%env SWANLAB_PROJ=CANN_SwanLab

def _swanlab_credential_saved() -> bool:
    """检查本地是否已有 SwanLab 凭证（默认存储在 ~/.netrc）"""
    netrc = Path.home() / '.netrc'
    if not netrc.exists():
        return False
    try:
        return 'swanlab.cn' in netrc.read_text()
    except Exception:
        return False

if _swanlab_credential_saved():
    print('检测到本地已有 SwanLab 凭证，跳过登录。')
else:
    api_key = getpass.getpass('请输入你的 SwanLab API Key（从 https://swanlab.cn/settings 获取）: ')
    swanlab.login(api_key=api_key, save=True)
    print('SwanLab 登录成功！凭证已保存到 ~/.netrc，下次运行无需再输入。')


swanlab_runs_logged = 0  # 累计真正记录成功的 run 数


def measure_step_time(model, batches, warmup=WARMUP, run_name='baseline'):
    """跑 len(batches) 个 step，返回稳定阶段平均 step time + 全部 loss。
    每步的 loss / step_time_ms 同步推到 SwanLab。"""
    global swanlab_runs_logged
    swanlab.init(
        project=os.environ['SWANLAB_PROJ'],
        experiment_name=f'SwanRmsNorm-{run_name}',
        config=dict(
            model=MODEL_PATH, batch_size=BATCH_SIZE, seq_len=SEQ_LEN,
            lora_r=LORA_R, optimizer='AdamW', lr=2e-4,
            variant=run_name,
        ),
        reinit=True,
    )

    model.train()
    opt = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=2e-4)

    times, losses = [], []
    for i, batch in enumerate(batches):
        torch_npu.npu.synchronize()
        t0 = time.perf_counter()
        opt.zero_grad(set_to_none=True)
        out = model(**batch)
        out.loss.backward()
        opt.step()
        torch_npu.npu.synchronize()
        step_ms = (time.perf_counter() - t0) * 1000
        loss_v  = float(out.loss.detach().item())
        times.append(step_ms)
        losses.append(loss_v)

        swanlab.log({
            'loss': loss_v,
            'step_time_ms': step_ms,
            'is_warmup': int(i < warmup),
        }, step=i)

    swanlab.finish()
    swanlab_runs_logged += 1

    stable = times[warmup:]
    return sum(stable) / len(stable), times, losses


# Baseline LoRA SFT
print('===== Baseline LoRA SFT =====')
m = build_lora_model()
n_train = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'  trainable params: {n_train:,}')
t_base, times_base, losses_base = measure_step_time(m, batches, run_name='baseline')
print(f'  step time: {t_base:.2f} ms')
print(f'  loss: first={losses_base[0]:.3f}  last={losses_base[-1]:.3f}  (期望下降)')
del m; gc.collect(); torch.npu.empty_cache()

# Patched LoRA SFT
print('\n===== Patched LoRA SFT (SwanRmsNorm) =====')
m = build_lora_model()
patch_qwen3_rmsnorm(m)
t_patched, times_patched, losses_patched = measure_step_time(m, batches, run_name='patched')
print(f'  step time: {t_patched:.2f} ms')
print(f'  loss: first={losses_patched[0]:.3f}  last={losses_patched[-1]:.3f}  (期望下降)')
del m; gc.collect(); torch.npu.empty_cache()

sft_speedup = t_base / t_patched
print('\n' + '═' * 50)
print(f'  Baseline step  : {t_base:.2f} ms')
print(f'  Patched  step  : {t_patched:.2f} ms')
print(f'  SFT Speedup    : {sft_speedup:.3f} ×')
print('═' * 50)
print(f'\n已记录 {swanlab_runs_logged} 个 SwanLab run（baseline + patched），在 swanlab.cn 上看 loss / step_time 对比。')

### §7.4 微调加速可视化

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt   # 显式 import：即使你跳过了 §6.1，本 cell 也能跑

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# 左图：每个 step 的 step time（看稳定性）
xs = np.arange(len(times_base))
ax1.plot(xs, times_base,    'o-', color='#9E9E9E', markersize=3, label='Baseline')
ax1.plot(xs, times_patched, 'o-', color='#4CAF50', markersize=3, label='Patched (SwanRmsNorm)')
ax1.axvspan(0, WARMUP - 0.5, alpha=0.15, color='red', label='warmup (excluded)')
ax1.set_xlabel('step')
ax1.set_ylabel('step time (ms)')
ax1.set_title(f'LoRA SFT step time (B={BATCH_SIZE}, S={SEQ_LEN})')
ax1.legend()
ax1.grid(alpha=0.3)

# 右图：平均 step time 对比
ax2.bar(['Baseline', 'Patched\n(SwanRmsNorm)'],
        [t_base, t_patched],
        color=['#9E9E9E', '#4CAF50'])
ax2.set_ylabel('avg step time (ms)')
ax2.set_title(f'LoRA SFT Speedup: {sft_speedup:.3f}×')
ax2.text(1, t_patched, f'{sft_speedup:.2f}×', ha='center', va='bottom',
         fontsize=14, fontweight='bold', color='#2E7D32')

plt.tight_layout()
plt.savefig('notebook_lora_sft.png', dpi=120)
plt.show()
print('保存到 notebook_lora_sft.png')

---

## §8 总结

你完成了一条完整的「把课程提供的 AscendC kernel 接到大模型」实操链路：

1. ✅ 读懂了 ~150 行 SwanRmsNorm AscendC kernel 的 fused 设计
2. ✅ 在 NPU 上完成编译、部署
3. ✅ **自己写**了 PyTorch 集成层：autograd.Function + monkey-patch（带工厂函数）
4. ✅ 推理对比：tokens/sec 端到端 **~1.30-1.55×** 加速（145 个 RMSNorm 模块全部接管）
5. ✅ LoRA SFT 对比：step time 端到端 **~1.05-1.15×** 加速，loss + step time 用 SwanLab 全记录

为什么单算子 ~7× 加速但端到端只有 1.5×？这是 **Amdahl 律**——RMSNorm 在整步推理只占 25%，所以 7× 局部 → 1.5× 全局；在 SFT 一步里只占 ~10%，所以只有 1.05-1.15× 端到端。

**详见指导书第九章总结与讨论。** 提交 `results/notebook_inference.png` 和 `results/notebook_lora_sft.png` 作为本次实训交付物，SwanLab 的 run 链接也可以一并提交。